# VC-Genome: Human-Phrase vs Vision Extraction Agreement

Compares the two Claude extractions for the same 520 images:

- **API (human-grounded):** `vc_genome_output_full/llm_extractions_api.json` — extracted from participant comments + curated phrases only (no image).
- **Vision (image-grounded):** `vc_genome_output_full/llm_extractions_vision.json` — extracted from the rendered image + comments.

We measure agreement at three layers:

1. **Objects** — canonical synsets (via `OBJECT_SYNSETS` fallback chain).
2. **Attributes** — `(object_synset, normalized_attribute)` pairs.
3. **Relationships** — full canonical triples *(strict)* and subject/object pairs *(loose)*.

Sentiment agreement is measured on the matched subset only.

**Not used in the matching:** `VisType`, `NormalizedVC`. Matching is purely set-theoretic over canonical O/A/R keys.

In [ ]:
import sys, json, pandas as pd, numpy as np
from pathlib import Path

sys.path.insert(0, 'scripts')
import match_api_vs_vision as m

OUT = Path('vc_genome_output_full/match')
print('Outputs:', list(OUT.glob('*')))

## 1. Run the matching pipeline

In [ ]:
m.main()

## 2. Per-image distribution of agreement

In [ ]:
per_image = pd.read_csv(OUT / 'per_image_metrics.csv')
print('Images:', len(per_image))
per_image.head(3)

In [ ]:
headline = ['obj_jaccard', 'obj_precision', 'obj_recall', 'obj_f1',
            'attr_jaccard', 'attr_f1',
            'rel_strict_jaccard', 'rel_strict_f1',
            'rel_loose_jaccard',  'rel_loose_f1',
            'attr_sentiment_agreement', 'rel_sentiment_agreement']
per_image[headline].describe().round(3).T

In [ ]:
import matplotlib.pyplot as plt

layer_bars = {
    'Objects (synset)':         'obj_f1',
    'Attributes (syn+attr)':    'attr_f1',
    'Rel. strict (s,p,o)':      'rel_strict_f1',
    'Rel. loose (s,o)':         'rel_loose_f1',
}

fig, ax = plt.subplots(figsize=(7, 3.2))
means = [per_image[c].mean() for c in layer_bars.values()]
stds  = [per_image[c].std()  for c in layer_bars.values()]
x = np.arange(len(layer_bars))
ax.bar(x, means, yerr=stds, capsize=4, color='#4C78A8')
ax.set_xticks(x); ax.set_xticklabels(list(layer_bars.keys()), rotation=15, ha='right')
ax.set_ylabel('F1 (vision vs human-phrase)')
ax.set_ylim(0, 1.0)
ax.set_title('Per-image agreement, mean ± sd (N={})'.format(len(per_image)))
for xi, mu in zip(x, means):
    ax.text(xi, mu + 0.02, f'{mu:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUT / 'layer_f1_bar.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, (title, col) in zip(axes, layer_bars.items()):
    ax.hist(per_image[col].dropna(), bins=20, color='#4C78A8', edgecolor='white')
    ax.axvline(per_image[col].mean(), color='crimson', lw=1.5, label=f'mean={per_image[col].mean():.2f}')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('F1'); ax.legend(fontsize=8)
axes[0].set_ylabel('# images')
plt.tight_layout()
plt.savefig(OUT / 'layer_f1_hist.png', dpi=150)
plt.show()

## 3. Sentiment agreement on matched elements

When the two sides name the *same* attribute or relationship, do they assign the same sentiment (+/−)?

In [ ]:
cols = ['attr_sentiment_agreement', 'attr_sentiment_n',
        'rel_sentiment_agreement',  'rel_sentiment_n']
per_image[cols].describe().round(3).T

## 4. Coverage of the canonicalization dictionary

In [ ]:
unmapped = pd.read_csv(OUT / 'unmapped_terms.csv')
print('Unmapped term occurrences by side × kind:')
display(unmapped.groupby(['side','kind'])['count'].agg(['sum','count']).rename(columns={'sum':'tokens','count':'types'}))

print('\nTop-15 unmapped objects (API side):')
display(unmapped[(unmapped.kind=='object') & (unmapped.side=='api')]
        .sort_values('count', ascending=False).head(15))

print('\nTop-15 unmapped predicates (API side):')
display(unmapped[(unmapped.kind=='predicate') & (unmapped.side=='api')]
        .sort_values('count', ascending=False).head(15))

## 5. Qualitative slice

Low / median / high agreement examples (by object-F1).

In [ ]:
print((OUT / 'qualitative_slice.md').read_text(encoding='utf-8')[:4000])